In [ ]:

!pip install faiss-cpu sentence-transformers groq

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
from groq import Groq

In [ ]:
Groq.api_key="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"

In [ ]:
docs=[
    "The Taj Mahal is a white marble mausoleum located in Agra, Uttar Pradesh. It was commissioned by the Mughal emperor Shah Jahan in memory of his wife Mumtaz Mahal. Construction began in 1632 and was completed in 1653. It is recognized as a UNESCO World Heritage Site.",
    "Kerala is known as God's Own Country due to its natural beauty. The state is famous for its backwaters, houseboats, beaches, and Ayurvedic treatments. Popular tourist destinations include Alleppey, Munnar, and Kochi. Tourism plays a major role in Kerala's economy.",
    "Jaipur, the capital city of Rajasthan, is known as the Pink City. It is famous for historical landmarks such as the Hawa Mahal, Amber Fort, and City Palace. Jaipur is part of India's Golden Triangle tourist circuit along with Delhi and Agra.",
    "Goa is India's smallest state by area and is famous for its beaches, nightlife, and Portuguese heritage. Popular beaches include Baga Beach, Calangute Beach, and Anjuna Beach. Goa attracts millions of domestic and international tourists every year.",
    "The Himalayan region offers opportunities for trekking, mountaineering, and adventure tourism. Popular destinations include Manali, Leh-Ladakh, Shimla, and Darjeeling. The region is known for its scenic landscapes, snow-capped mountains, and unique cultural experiences."

]


In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
doc_embeddings=model.encode(docs).astype("float32")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
doc_embeddings.shape

(5, 384)

In [ ]:
dimension = doc_embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [ ]:
query="Which places are famous tourist destinations in Kerala?"
query_embedding=model.encode([query]).astype("float32")
top_k=2
distances, indices = index.search(query_embedding,top_k)

retrieved_chunks=[docs[i] for i in indices[0]]
print("Retrieved Chunks🔎:")
for chunk in retrieved_chunks:
  print("-",chunk)

Retrieved Chunks🔎:
- Kerala is known as God's Own Country due to its natural beauty. The state is famous for its backwaters, houseboats, beaches, and Ayurvedic treatments. Popular tourist destinations include Alleppey, Munnar, and Kochi. Tourism plays a major role in Kerala's economy.
- Goa is India's smallest state by area and is famous for its beaches, nightlife, and Portuguese heritage. Popular beaches include Baga Beach, Calangute Beach, and Anjuna Beach. Goa attracts millions of domestic and international tourists every year.


In [ ]:
from groq import Groq


client = Groq(api_key="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")

context = "\n".join(retrieved_chunks)

prompt = f"""
Answer the question based only on the context below.

Context:
{context}

Question:
{query}

Answer:
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",  # or llama3-8b-8192
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0.2
)

print("Final Answer:\n")
print(response.choices[0].message.content)

Final Answer:

Alleppey, Munnar, and Kochi.
